# L37 - Simulation Optimization by Grid Search

**Learning objectives**
- Formulate a simulation optimization problem over a finite design set.
- Run a coarse search over `(s, S)` policies using `simdes`.
- Report uncertainty with replication-based confidence intervals.
- Identify which policies deserve more evaluation budget.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from simdes.analysis import confidence_interval
from simdes.models import SSInventory

## Coarse search over candidate policies

We start with a small set of candidate `(s, S)` pairs. The goal is not to find the globally optimal policy in one pass. The goal is to learn which regions of the decision space are promising enough to justify more replications.

In [ ]:
def evaluate_policy(s: int, S: int, n_reps: int = 12, sim_time: float = 365.0, base_seed: int = 2026):
    model = SSInventory(reorder_point=s, order_up_to=S, sim_time=sim_time, seed=base_seed)
    df = model.run_replications(n_reps)
    costs = df['avg_total_cost'].to_numpy()
    mean_cost, ci_lo, ci_hi = confidence_interval(costs)
    return {
        's': s,
        'S': S,
        'mean_cost': mean_cost,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi,
        'half_width': 0.5 * (ci_hi - ci_lo),
    }

candidates = [(10, 80), (20, 100), (30, 120), (40, 140), (50, 160)]
results = pd.DataFrame([evaluate_policy(s, S) for s, S in candidates]).sort_values('mean_cost')
results

In [ ]:
labels = [f"({row.s}, {row.S})" for row in results.itertuples()]
means = results['mean_cost'].to_numpy()
errors = np.vstack([means - results['ci_lo'].to_numpy(), results['ci_hi'].to_numpy() - means])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, means, color='tab:blue', alpha=0.8)
ax.errorbar(labels, means, yerr=errors, fmt='none', ecolor='black', capsize=4)
ax.set_ylabel('Average total cost')
ax.set_xlabel('(s, S) policy')
ax.set_title('Coarse simulation optimization search')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()

## Try It Yourself

1. Add at least four more candidate `(s, S)` policies.
2. Double the number of replications for the two best candidates.
3. Explain whether the ranking changes once the confidence intervals tighten.